### Step-by-step Flow

1. Upload a PDF file.
2. Load the document using a document loader.
3. Combine document pages into a single text.
4. Split the text into chunks.
5. Convert chunks into `Document` objects.
6. Generate embeddings and store them in a vector store (FAISS).
7. Perform similarity search and retrieval.
8. Pass retrieved context to an LLM using a prompt.
9. Build a simple chatbot on top of the retrieval pipeline.


## 1. Installing Required Libraries

In [1]:
!pip install -q langchain-community langchain-google-genai faiss-cpu pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


##2. File Upload (Google Colab)

In [2]:
from google.colab import files

uploaded = files.upload()

Saving FYP Proposals List.pdf to FYP Proposals List.pdf


In [3]:
print(uploaded)

{'FYP Proposals List.pdf': b'%PDF-1.7\n\n4 0 obj\n(Identity)\nendobj\n5 0 obj\n(Adobe)\nendobj\n8 0 obj\n<<\n/Filter /FlateDecode\n/Length 93022\n/Length1 374416\n/Type /Stream\n>>\nstream\nx\x9c\xec}\x07`TU\xda\xf6{\xee\xbd\xd3\xdb\x9d\xde3}\x92\xcc\xa4\x91F\x02!\x994z\xe8J\x82DB\x15\x14\x91&\xd8EWE\xb1\xb1\xba+b\xc5\xee\xea\xee:L,\x01,XV\xdd\xb5\xa1\xae\xab\xab~\x8a\xcaZVQv\x17YWa\xe6\x7f\xcf\x9d@\xc2g\xfe\xfdD\xc0\x00\x9egr\x9eS\xef\xb9\xef=\xe5=%\xb7\x00\x01\x00\'\x92\x00\xab\x9b&\x8e\x18v\xea\x9b\xb7\x9bA\xb6z\x1b@\xe4\xb7\xc3\x9a\x9a\x87\x86\xf5\xf9\x01\xe0\xf3\xe6\x03\xf0\x89a\xe3\xc6N\x9c\xb6\xeb\xaa2\xe0\xe3\xbf\x05\xf2\xf1\xf0a\x13\x8fk\x08\x9e\xb4}\x02\xc8\x8c;\x01\xce[0r\xe2\xa4\xa1\xa7\xc6\xe6\xc9\xf1\xf8Y\x98k\xce\xe8I\x13\x87\xef\x0c\xb6~\x0e\xd0T\x02`:w\xec\xc4\xe2R\xd7\xd9Sn\x01\xe0\x1c\x18\xdf1\xaeq\xf4\xa4s\xec\x17}\x8b\xf9\x97\xa3\xbf\xf2\xf8\xa6\x96\xd6\x89\xbf_`\x028>\t`\xfc\xd5\xccS\xa7/|\xe4I_\x0bp\xa1/\x00\xc8\xce\x99\xcb\x96\xfa\x97~\xf8\x9f\x9b\x81\xab\xbb\x1

In [7]:
filename = list(uploaded.keys())[0]

print(filename)

FYP Proposals List.pdf


##3. Document Loaders

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader


loader = PyMuPDFLoader(filename)

docs = loader.load()

In [9]:
print(docs)

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': '', 'creationdate': '2025-09-12T00:54:51+05:00', 'source': 'FYP Proposals List.pdf', 'file_path': 'FYP Proposals List.pdf', 'total_pages': 5, 'format': 'PDF 1.7', 'title': 'Microsoft Word - FYP Proposals', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-12T00:54:51+05:00', 'trapped': '', 'modDate': "D:20250912005451+05'00'", 'creationDate': "D:20250912005451+05'00'", 'page': 0}, page_content='Govt. Municipal Graduate College, Jaranwala Road, Faisalabad \n(Affiliated with GC University Faisalabad) \nCourse: CSI-630 Project (Proposal), 0-2 credit hours (Practical) \nSemester: BS Computer Science, 7th Semester \nInstructor: Muhammad Tehseen Qureshi \nDate: September 12, 2025 \nDear Students, \nI hope you are excited to begin working on your final-year project (FYP) proposals for CSI-630! \nAttached is the list of assigned project topics and corresponding tools/methods for each of you. \nYou are required

##4. Combining Document Content

In [10]:
full_text = ""

for doc in docs:
  full_text += doc.page_content + "\n"

print(full_text)

Govt. Municipal Graduate College, Jaranwala Road, Faisalabad 
(Affiliated with GC University Faisalabad) 
Course: CSI-630 Project (Proposal), 0-2 credit hours (Practical) 
Semester: BS Computer Science, 7th Semester 
Instructor: Muhammad Tehseen Qureshi 
Date: September 12, 2025 
Dear Students, 
I hope you are excited to begin working on your final-year project (FYP) proposals for CSI-630! 
Attached is the list of assigned project topics and corresponding tools/methods for each of you. 
You are required to work on the topic assigned to you as per the provided list. 
Group Work Option: 
If you wish to work in a group, you may form a group of two students only. Groups must select 
one of the two assigned topics of the group members to work on. The tools and methods specified 
for that topic must be followed. You cannot choose a topic other than the one assigned to you or 
your group partner. 
Action Required: 
 
If you want to work in a group, finalize your group (maximum 2 students) by

##5. Text Splitters

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 10
)

text_chunks = splitter.split_text(full_text)

print(text_chunks)
print(len(text_chunks))

['Govt. Municipal Graduate College, Jaranwala Road, Faisalabad \n(Affiliated with GC University Faisalabad) \nCourse: CSI-630 Project (Proposal), 0-2 credit hours (Practical) \nSemester: BS Computer Science, 7th Semester \nInstructor: Muhammad Tehseen Qureshi \nDate: September 12, 2025 \nDear Students,', 'I hope you are excited to begin working on your final-year project (FYP) proposals for CSI-630! \nAttached is the list of assigned project topics and corresponding tools/methods for each of you. \nYou are required to work on the topic assigned to you as per the provided list. \nGroup Work Option:', 'If you wish to work in a group, you may form a group of two students only. Groups must select \none of the two assigned topics of the group members to work on. The tools and methods specified \nfor that topic must be followed. You cannot choose a topic other than the one assigned to you or', 'your group partner. \nAction Required: \n\uf0b7 \nIf you want to work in a group, finalize your gr

##6. Converting Text Chunks into Document Objects

In [12]:
from langchain_core.documents import Document

chunks = [Document(page_content=chunk) for chunk in text_chunks]

print(chunks)

[Document(metadata={}, page_content='Govt. Municipal Graduate College, Jaranwala Road, Faisalabad \n(Affiliated with GC University Faisalabad) \nCourse: CSI-630 Project (Proposal), 0-2 credit hours (Practical) \nSemester: BS Computer Science, 7th Semester \nInstructor: Muhammad Tehseen Qureshi \nDate: September 12, 2025 \nDear Students,'), Document(metadata={}, page_content='I hope you are excited to begin working on your final-year project (FYP) proposals for CSI-630! \nAttached is the list of assigned project topics and corresponding tools/methods for each of you. \nYou are required to work on the topic assigned to you as per the provided list. \nGroup Work Option:'), Document(metadata={}, page_content='If you wish to work in a group, you may form a group of two students only. Groups must select \none of the two assigned topics of the group members to work on. The tools and methods specified \nfor that topic must be followed. You cannot choose a topic other than the one assigned to y

##7. Vector Stores

In [14]:
from google.colab import userdata
gemini_key = userdata.get('GEMINI')

In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001",api_key = gemini_key,output_dimensionality=16)

vector_store = FAISS.from_documents(documents=chunks,embedding=embedding_model)



##8. Similarity Search (Testing Vector Store)



In [19]:
results = vector_store.similarity_search("What is best project",k=6)

print(results)

[Document(id='c7587fd3-4207-4570-a542-1fc6e96f6c0b', metadata={}, page_content='Privacy-preserving \ndata sharing using \nsecure multi-party \ncomputation\nPython, MP-SPDZ, \nPySyft \n25 182 \n108788 \nSAIM ASHRAF \nZero-trust architecture \nfor enterprise network \nsecurity\nPython, OpenZiti, \nDocker, Kubernetes \n26 184 \n108790 \nMUHAMMAD \nQOSAIN HAIDER\nAutomated \nvulnerability detection'), Document(id='ef55752d-226b-4db8-9f4b-5a607b3002c4', metadata={}, page_content='in open-source \nsoftware\nPython, Bandit, \nSonarQube, GitHub API\n27 185 \n108791 \nMUHAMMAD \nAWAIS \nHomomorphic \nencryption for secure \ncloud computing\nPython, TenSEAL, \nMicrosoft SEAL \n28 186 \n108792 \nHAFIZ NOUMAN \nRAZA \nCyber threat \nintelligence using \npredictive analytics\nPython, Scikit-learn,'), Document(id='039cd23f-cc6e-4fc1-ace5-1164ddf81457', metadata={}, page_content='Project Topic \nTools \n35 195 \n108799 \nMUHAMMAD \nUZAIR \nPredictive analytics \nfor urban traffic \nmanagement\nPython

##9. Retriever

In [20]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [22]:
results = retriever.invoke("extract all topic related to deep learning/machine learning ")

print(results)

[Document(id='f3773205-c93e-4013-aec2-6e7902ae19ac', metadata={}, page_content='I hope you are excited to begin working on your final-year project (FYP) proposals for CSI-630! \nAttached is the list of assigned project topics and corresponding tools/methods for each of you. \nYou are required to work on the topic assigned to you as per the provided list. \nGroup Work Option:'), Document(id='cb54b6a2-3b3a-4fa6-ad83-efcbf14da3de', metadata={}, page_content='robustness\nPython, Scikit-learn, \nXGBoost, LightGBM \n21 177 \n108783 \nMUHAMMAD \nREHAN \nPost-quantum \ncryptography for \nsecuring data against \nquantum attacks\nPython, Qiskit, \nCryptography Library \n(PQCrypto) \n22 178 \n108784 \nLAIBA FARHEEN\nBlockchain-based \nsecure authentication \nfor IoT devices'), Document(id='6826865c-96d2-4bb9-a9e7-c6049096098a', metadata={}, page_content='Govt. Municipal Graduate College, Jaranwala Road, Faisalabad \n(Affiliated with GC University Faisalabad) \nCourse: CSI-630 Project (Proposal), 

##10. Preparing Context for the LLM

In [23]:
def get_context_string(documents):
  context_text= ""
  for doc in documents:
    context_text+= doc.page_content + "\n"

  return context_text

In [24]:
context= get_context_string(results)

print(context)

I hope you are excited to begin working on your final-year project (FYP) proposals for CSI-630! 
Attached is the list of assigned project topics and corresponding tools/methods for each of you. 
You are required to work on the topic assigned to you as per the provided list. 
Group Work Option:
robustness
Python, Scikit-learn, 
XGBoost, LightGBM 
21 177 
108783 
MUHAMMAD 
REHAN 
Post-quantum 
cryptography for 
securing data against 
quantum attacks
Python, Qiskit, 
Cryptography Library 
(PQCrypto) 
22 178 
108784 
LAIBA FARHEEN
Blockchain-based 
secure authentication 
for IoT devices
Govt. Municipal Graduate College, Jaranwala Road, Faisalabad 
(Affiliated with GC University Faisalabad) 
Course: CSI-630 Project (Proposal), 0-2 credit hours (Practical) 
Semester: BS Computer Science, 7th Semester 
Instructor: Muhammad Tehseen Qureshi 
Date: September 12, 2025 
Dear Students,



##11. LLM and Prompt Template

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

llm  = ChatGoogleGenerativeAI(model="gemini-2.5-flash",api_key = gemini_key)

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful cricket expert.
Answer ONLY using the following context from the uploaded PDF.
If the context is insufficient, just say "I don't know".

Context:
{context}

Question:
{question}
"""
)

In [ ]:
chain = prompt | llm

In [ ]:
query = "Who is Salman Khan"

retrieved_docs = retriever.invoke(query)


final_retrieved_docs = get_context_string(retrieved_docs)

print(final_retrieved_docs)


response = chain.invoke({
    "question": query,
    "context": final_retrieved_docs
})


print(response.content)

Twenty20 (T20): 20 overs per side.
🧢 2. Player Roles
Batsmen: Aim to score runs.
Bowlers: Aim to dismiss batsmen and restrict runs.
Fielders: Support bowlers by catching the ball and stopping runs.
Wicketkeeper: Specialized fielder behind the stumps.
⚖️ 3. Scoring Rules
1
Bye: Runs taken without touching the bat.
Leg bye: Runs taken after ball hits the batsman’s body.
🚫 4. Dismissals (Ways to Get Out)
1. Bowled
2. Caught
3. Leg Before Wicket (LBW)
4. Run Out
5. Stumped
6. Hit Wicket
7. Handled the Ball (obsolete in some formats)
8. Obstructing the Field
Cricket Laws
🏏 1. Basics of the Game
Objective: Score more runs than the opponent.
Teams: Two teams of 11 players each.
Innings: Each team gets one or two innings depending on the format.
Formats:
Test Cricket: Up to 5 days, unlimited overs.
One Day International (ODI): 50 overs per side.

I don't know.
